# DEM 实际高程剖面提取

**用途：** 从 DEM 中选择矩形区域，沿横向、纵向或任意两点提取一段真实高程。
默认按源文件声明的米制平面坐标计距，不猜测 EPSG，不减去海拔均值。

流程：**安装依赖 → 修改路径 → 查看 DEM → 选择范围与方向 → 更新剖面 → 导出**。
最后导出的 `surface_profile.csv` 仅两列 `x_m,elevation_m`，可供二维波场程序直接导入。

同目录需保留 `dem_profiles.py` 和 `profile_selector.py`。当前 DEM 是 GDAL 不能直接识别的投影代码 9 固定记录变体，
模块已经加入校验读取分支，原始文件保持只读。

## 1. 首次安装依赖

选择 Python 3.10+ 内核。若提示缺少模块，取消下一单元安装命令的注释后运行一次，再重启内核。
如已有依赖，可以直接继续。VS Code 和 JupyterLab 均可使用；鼠标框选额外需要 ipympl 前端支持。

In [ ]:
# 首次使用时，取消注释；安装在当前 Notebook 内核对应的 Python 环境。
# %pip install --index-url https://pypi.org/simple "numpy>=1.24" "pandas>=2" "matplotlib>=3.7,<4" "rasterio>=1.4.4,<2" "pyproj>=3.6,<4" "ipywidgets>=8,<9" "ipympl>=0.9,<1"

In [ ]:
from pathlib import Path
import os, sys, json
sys.dont_write_bytecode = True  # 不向仓库写入 __pycache__，生成结果仍保存在外层 out。

# 兼容 VS Code 从工作区目录、或 Jupyter 从 Notebook 所在目录启动。
module_dir = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidate in (base, base/'tools/dem_profiles', base/'SeisWaveLab_Curvilinear/tools/dem_profiles'):
        if (candidate/'dem_profiles.py').is_file():
            module_dir = candidate.resolve()
            break
    if module_dir is not None:
        break
if module_dir is None:
    raise FileNotFoundError('请把 Notebook 和 dem_profiles.py 放在同一目录，并从该目录启动。')
sys.path.insert(0, str(module_dir))
from dem_profiles import (DEM, project_output, line_in_roi, extract_profile,
                  longest_valid_segment, plot_map, plot_profile, export_profile)

OUTPUT_DIR = project_output()       # 项目内运行时，自动使用外层 out/dem_profiles
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR/'cache/matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
%matplotlib inline

## 2. 输入路径与坐标约定

修改 `DEM_PATH` 即可换文件。对当前文件，无需填写坐标系和单位覆盖参数。
`grid` 表示源文件米制投影平面中的水平距离；只有以后确认 CRS 后，才考虑 `geodesic`。
未知单位会报错，不会自动把角度当米。

In [ ]:
DEM_PATH = Path(r'D:/Data file/航飞ECW/导出区域数据/3_jingbiancenter.dem')
CRS_OVERRIDE = None             # 不猜测 EPSG；仅在确认后填写，例如 'EPSG:xxxx'
XY_UNIT_OVERRIDE = None         # 当前文件声明水平单位为米，保持 None
Z_UNIT_OVERRIDE = None          # 当前文件声明垂直单位为米，保持 None
DISTANCE_MODE = 'grid'          # 默认米制格网距离，不作经纬度或垂直基准转换

if 'selector_ui' in globals():
    selector_ui.close()
if 'dem' in globals():
    dem.close()
dem = DEM(DEM_PATH, output_dir=OUTPUT_DIR, crs_override=CRS_OVERRIDE,
            xy_unit_override=XY_UNIT_OVERRIDE, z_unit_override=Z_UNIT_OVERRIDE)
info = dem.info()
display(pd.DataFrame({
    '项目': ['源文件', '读取方式', '行×列', '像元间距（源单位）', '高程单位', '水平每单位米数', '确认的 CRS', '输出目录'],
    '值': [str(DEM_PATH), '校验固定记录兼容分支' if info['fallback_reader'] else 'Rasterio/GDAL',
            str(info['shape']), str(info['resolution']), str(info['height_unit']),
            str(info['horizontal_metres_per_unit']), str(info['crs']), str(OUTPUT_DIR)]
}))
print('注意：更密的输出采样属于插值，不会增加原始 DEM 的实际地形分辨率。')

## 3. 地图定位入口

交互地图集中在第 4 节。只有该地图支持点选、拖框、平移和缩放；高程曲线为静态图件。


In [ ]:
# Avoid a duplicate static map that looks interactive but cannot receive clicks.
print('请在下一节的交互地图中点选；这里不再重复显示静态 DEM 图片。')


## 4. 一维高程剖面选取与导出

常用参数与地图并排，窄窗口自动换行。精确坐标、截取终点、起点微移和插值方式在“精确坐标、终点与微调”中，任意两点的端点输入按需显示。

- X/Y 起点偏移按 DEM 左下像元中心计，移动时保留区域大小。
- 剖面位置按米调整；截取起点、长度和终点同步，导出使用当前参数。
- 深蓝线条、浅绿 / 浅蓝填充以参考高程分区，最高 / 最低点可关闭。
- “曲线高程”可切换实际高程、相对起点、相对样点均值。**仅改变图的显示，CSV 保留实际高程**。默认填充参考为起点；首点无效时使用首个有效样点并明确说明。
- 自然地形没有统一的峰谷宽度定义，因此不自动加宽度箭头。无效样点保持空缺，导出规则不变。
- 滑条释放后自动更新，关闭自动预览时点击“更新剖面”；图下样点表默认折叠。
- **鼠标定位**：只在下方标有“交互地图”的区域点选。切换点选模式会退出工具栏导航；也可点击“恢复点选”退出平移 / 缩放。静态曲线图不能用于定位。
- **DEM 局部范围**：自动模式使用已缩放且覆盖选区中心的视口，否则使用选区；也可明确选“当前地图视口”或“选取矩形”。`dem_zoom` 保存该范围，`dem_view` 另存当前视口快照。未启用交互画布时明确回退到选区绘制。
- **稳定导出**：先冻结当前数据和视口，在独立画布导出，避免影响交互地图；全部成功后才生成正式结果目录。失败的中间结果带 `.incomplete` 后缀及错误说明。
- **统一结果文件夹**：一次导出将 DEM 全图、视口截图、曲线、可选曲线局部图、CSV、参数和说明放入同一目录。图件使用白底、细线、Times New Roman 和 jet 色标，提供 600 dpi PNG 与 PDF / SVG。
- “高程曲线局部放大（可选）”用于另行导出曲线 `profile_zoom.png` 和 `profile_zoom.svg`，与 DEM 地图导出分开。范围按截取后剖面计距，保持全剖面的高程基准，CSV 不变。


更新模块后建议重启内核并全部运行。该单元会重载绘图和界面模块，无需 `%matplotlib widget`。导出同时提供 PNG 和 SVG，显示参考记录在 `selection.json`。

二维区域数组已独立至第 9 节，地图、选区、采样设置与导出按钮互不关联。本节只导出剖面及对应图件。


In [ ]:
if 'selector_ui' in globals():
    selector_ui.close()
import importlib, dem_profiles, profile_selector
importlib.reload(dem_profiles)
importlib.reload(profile_selector)
DEMProfileSelector=profile_selector.DEMProfileSelector
selector_ui = DEMProfileSelector(dem, OUTPUT_DIR, DISTANCE_MODE, enable_mouse=True)
display(selector_ui.widget)
selector_ui.refresh()

# 保留简洁变量，供下面的纯代码和批量示例使用。
roi_x,roi_y=selector_ui.roi_x,selector_ui.roi_y
direction,position=selector_ui.direction,selector_ui.position
spacing,start_dist,end_dist=selector_ui.spacing,selector_ui.start_dist,selector_ui.end_dist
interpolation,trim_invalid=selector_ui.interpolation,selector_ui.trim_invalid
x0,y0,x1,y1=selector_ui.x0,selector_ui.y0,selector_ui.x1,selector_ui.y1
state=selector_ui.state
refresh_profile=selector_ui.refresh
save_current_profile=selector_ui.export
left,bottom,right,top=dem.bounds

## 5. 地图点选与框选

第 4 节已经尝试创建交互地图，选择上方模式即可：

1. **单击定位起点**：横向沿 X、纵向沿 Y 将剖面起点放在点击位置；区域和剖面位置同步，接近边界时会限制在 DEM 内。
2. **拖框选择区域**：左键拖矩形，更新区域起点和大小。
3. **点选两个端点**：依次单击起点、终点，自动切到任意两点并选中整条线，随后仍可用起点/长度滑条截取。
4. **关闭点选**：只用滑条或数值定位。

地图工具栏处于平移/缩放模式时，不会同时执行点选；完成缩放后应关闭相应工具。
黑点标出当前截取起点，红线标出将要导出的段。该地图直接创建 ipympl 画布，
不依赖可能未注册的 `widget` 后端别名，也不会切换已有静态预览的绘图后端。

若只见 `Loading widget`，请确认当前内核装有 ipympl/ipywidgets，重启内核并全部运行；
随后点击“启用 / 重建交互地图”。滑条定位无需交互地图即可使用。

In [ ]:
# 如前端重新连接，运行本单元也可重建地图；滑条和当前选择保留。
# selector_ui.enable_mouse(force=True)
print('当前内核：', sys.executable)
print('交互画布：', '已创建' if selector_ui.canvas is not None else '未创建，请查看控件中的提示')

## 6. 可选：纯代码精确截取

不使用控件时可直接写端点或 ROI。例如把下面的开关改为 True，可沿当前 ROI 的纵向中线，
从 100 m 截至 900 m，以 10 m 间距输出一条 800 m 剖面。前提是完整线长至少 900 m。
NoData 不会被填成零，默认拒绝导出；需要时请明确调用 `longest_valid_segment`。

In [ ]:
RUN_MANUAL_EXAMPLE = False
if RUN_MANUAL_EXAMPLE:
    roi = (*roi_x.value,*roi_y.value)
    p0,p1 = line_in_roi(roi,direction='vertical',position=.5)
    manual_table,manual_metadata = extract_profile(dem,p0,p1,spacing_m=10,
                                                   start_m=100,end_m=900,
                                                   method='bilinear',distance_mode='grid')
    manual_folder = export_profile(dem,manual_table,manual_metadata,OUTPUT_DIR,roi)
    display(manual_table.head())
    print(manual_folder)

## 7. 可选：批量提取横向和纵向测线

将下面的开关改为 True，会在当前矩形的 25%、50%、75% 位置各提取横向和纵向剖面，
分别保存到独立文件夹，坐标和参数随数据保存。

In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    roi = (*roi_x.value,*roi_y.value)
    for mode in ('horizontal','vertical'):
        for fraction in (.25,.5,.75):
            p0,p1 = line_in_roi(roi,mode,fraction)
            table,metadata = extract_profile(dem,p0,p1,spacing_m=spacing.value,
                                               method=interpolation.value,distance_mode=DISTANCE_MODE)
            metadata.update(batch_direction=mode,batch_position=fraction)
            print(mode,fraction,export_profile(dem,table,metadata,OUTPUT_DIR,roi))

## 8. 导入二维波场程序

1. 在“外部地表高程”中选择导出的 `surface_profile.csv`，载入后检查覆盖范围。
2. **实际海拔数据保留原值。** 通常保留导入器自动计算的高程均值，作为模型局部坐标基准；
   不要把人工零高程示例的基准 0 m 直接套到约 1000 多米的实际海拔上。
3. 需满足 `(Nx−1)h ≤ 剖面长度`。恰好 1000 m 时，可从 Nx=401、h=2.5 m 开始设置，
   Nz、地形展平深度、主频和物性另外检查。DEM 输出间距与波场网格间距是不同参数。
4. “完成”后“重置并应用参数”，核对生效网格与 dt。

输出保留实际高程，无归一化、无去趋势、无自动平滑。它反映源 DEM，并不表示已验证源测绘数据的绝对精度。
操作结束后可手动运行 `dem.close()` 释放文件；操作控件期间请保持数据集打开。

更多格式说明见同目录 README；技术接口依据：
[Rasterio](https://rasterio.readthedocs.io/en/stable/topics/reading.html)、
[GDAL USGSDEM](https://gdal.org/en/stable/drivers/raster/usgsdem.html)、
[PyProj](https://pyproj4.github.io/pyproj/stable/api/crs/crs.html)、
[Matplotlib RectangleSelector](https://matplotlib.org/stable/gallery/widgets/rectangle_selector.html)。

## 9. 二维区域选取与导出（独立 UI）

本面板使用自己的交互地图与矩形选区，**不读取第 4 节的剖面位置或选区**。拖动鼠标框选后，可输入精确的 X 宽度、Y 长度及左下角偏移（米）。平移/缩放后点击“恢复框选”；也可直接输入数值，不依赖鼠标事件。

- **按点数**：设置 X 点数，按同一个间距自动计算 Y 点数，保留精确的矩形长宽，不再居中裁成正方形。
- **按间距**：输入 X/Y 共用间距（米），自动计算行列数。不追加短间隔，以兼容三维等间距格式。
- 默认**不含末端（L/N）**：1000 m 宽、500 点就是 2 m 间距；样点为 0,2,…,998 m，节点跨度 998 m，不能将其误写成 1000 m。1 km × 800 m、2 m 间距对应 500 × 400 点。
- 可选**包含末端（L/(N−1)）**：1 km、2 m 间距对应 501 点；按 500 点则间距为 1000/499 m。长度不能整除固定间距时不追加终点。

每次点击“导出二维区域”生成一个独立文件夹，包含所选 `.vti` / `.vts`、无表头高程矩阵 CSV、采样元数据 JSON 及 jet 配色、下置横向色标的 PNG/PDF/SVG 区域图。原始高程不归一化；行沿 Y 增大，列沿 X 增大。NoData 拒绝整块导出，加密采样不提高源 DEM 实测分辨率。

下面默认建立居中 1 km × 1 km、300 × 300 点测试集，间距为 1000/300 m。`RUN_REGION_TEST=False` 可关闭自动测试导出，仍可使用面板。导入后的求解网格继续按总预算动态分配 XYZ。

导出格式仅保留 **VTI / VTS**，默认两者同时导出，也可单选。VTI 为单层规则 XY 图像，`Elevation` 点标量保存实际高程；VTS 的点坐标为原始平面 X/Y 和向上的实际高程 Z，可直接查看起伏地表。VTI 可在 ParaView 中按 `Elevation` 使用 Warp By Scalar 显示同一地形。三维程序读取两者时均转换为 `Z=基准−高程`，行列方向不变。

文件名包含点数及采样间隔，例如 `surface_grid_500x400_h2m.vti` / `.vts`；CSV、JSON 和区域图也使用同一前缀。完整数值精度保留在 VTK / JSON 中，不从文件名反推间距。三维地表导入仅接受单层、平面等间距、米制的 ASCII VTI/VTS，三维模型体和二进制 / 压缩数据不作为本接口输入。


In [ ]:
if 'region_ui' in globals():
    region_ui.close()
import importlib, region_selector
importlib.reload(region_selector)
region_ui = region_selector.DEMRegionSelector(
    dem, OUTPUT_DIR / 'regions', enable_mouse=True)
display(region_ui.widget)

RUN_REGION_TEST = True
if RUN_REGION_TEST:
    region_test_folder = region_ui.export()
    if region_test_folder is None:
        raise RuntimeError('二维区域测试导出失败，请查看上方面板提示。')
    region_test_info = json.loads(next(region_test_folder.glob('*_metadata.json')).read_text(encoding='utf-8'))
    print('测试数组：', region_test_info['nx'], '×', region_test_info['ny'], '；间距：', region_test_info['spacing_m'], 'm')
    for filename in region_test_info['vtk_files']:
        print('三维导入：', region_test_folder / filename)
